# 🌍 ESG RL Training — GRPO + Unsloth

**OpenEnv Hackathon Submission: ESG Compliance & Sustainability Environment**

This notebook trains an LLM to make better ESG (Environmental, Social, Governance)
decisions using Group Relative Policy Optimization (GRPO) with verifiable rewards.

**Stack:** Unsloth + TRL (GRPO) + ESGEnvironment

**Runtime:** Use a T4 GPU (Google Colab Free Tier works)

---

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Install dependencies
!pip install -q unsloth trl pydantic openai datasets peft accelerate bitsandbytes PyYAML
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
# Clone or mount your project
# Option A: Clone from GitHub (replace with your repo URL)
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/open_ENV.git'
!git clone {GITHUB_REPO} /content/open_ENV
%cd /content/open_ENV

# Option B: If using Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/open_ENV

## 2. Smoke Test (Validate Setup Without Training)

In [ ]:
# Verify environment and reward functions work
!python train_rl.py --smoke_test

## 3. Run Baseline Benchmark (Pre-Training Scores)

In [ ]:
# Run heuristic baseline to establish pre-training score floor
!python benchmark.py --mode random --output results/baseline_random.json
!python benchmark.py --mode heuristic --output results/baseline_heuristic.json
print('Baseline scores saved.')

## 4. Generate Training Dataset

In [ ]:
!python dataset_builder.py --episodes 20 --output data/esg_prompts.jsonl
# Check dataset size
import json
with open('data/esg_prompts.jsonl') as f:
    lines = f.readlines()
print(f'Dataset: {len(lines)} training samples')

## 5. Train with GRPO

In [ ]:
# Start with easy task, 200 steps
# Adjust --max_steps based on time budget
!python train_rl.py \
    --config train_config.yaml \
    --task basic_compliance \
    --max_steps 200 \
    --output_dir outputs/esg_rl_v1

## 6. Post-Training Benchmark (Compare vs. Baseline)

In [ ]:
!python benchmark.py \
    --model outputs/esg_rl_v1/lora_adapter \
    --output results/trained_v1.json
print('Post-training scores saved.')

## 7. Visualize Results

In [ ]:
!python plot_results.py \
    --baseline results/baseline_heuristic.json \
    --trained results/trained_v1.json \
    --output results/comparison.png

from IPython.display import Image
Image('results/comparison.png')

## 8. Save to HuggingFace Hub (Optional)

Once HF credits are available:

In [ ]:
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')

# model.push_to_hub('YOUR_USERNAME/esg-rl-agent')
# tokenizer.push_to_hub('YOUR_USERNAME/esg-rl-agent')
print('Uncomment above lines and add your HF token to push the model.')